<a href="https://colab.research.google.com/github/jc020230/practiceai/blob/main/3%EC%9E%A5_5_handling_data_issues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Handling duplicate, missing, or invalid data

## About the data
In this notebook, we will using daily weather data that was taken from the [National Centers for Environmental Information (NCEI) API](https://www.ncdc.noaa.gov/cdo-web/webservices/v2) and altered to introduce many common problems faced when working with data.

*Note: The NCEI is part of the National Oceanic and Atmospheric Administration (NOAA) and, as you can see from the URL for the API, this resource was created when the NCEI was called the NCDC. Should the URL for this resource change in the future, you can search for "NCEI weather API" to find the updated one.*

## Background on the data

Data meanings:
- `PRCP`: precipitation in millimeters
- `SNOW`: snowfall in millimeters
- `SNWD`: snow depth in millimeters
- `TMAX`: maximum daily temperature in Celsius
- `TMIN`: minimum daily temperature in Celsius
- `TOBS`: temperature at time of observation in Celsius
- `WESF`: water equivalent of snow in millimeters

Some important facts to get our bearings:
- According to the National Weather Service, the coldest temperature ever recorded in Central Park was -15°F (-26.1°C) on February 9, 1934: [source](https://www.weather.gov/media/okx/Climate/CentralPark/extremes.pdf)
- The temperature of the Sun's photosphere is approximately 5,505°C: [source](https://en.wikipedia.org/wiki/Sun)

## Setup
We need to import `pandas` and read in the dirty data to get started:

**요약: 데이터 문제 3종류와 대처법**

- 중복(duplicate): `duplicated()`로 찾고 → `drop_duplicates()`로 제거
- 빈칸(NaN / null): `isna()`로 찾고 → `dropna()`로 지우거나 / `fillna()`로 채우거나 / `interpolate()`로 추정
- 이상한 값(5505, -40, inf 같은 자리채움용 가짜 값): `describe()`로 발견 → `replace()`로 NaN으로 바꾸거나 `clip()`으로 범위 제한
- 제일 중요한 교훈: "할 수 있다 ≠ 해야 한다". 지우거나 채우기 전에 항상 데이터 의미를 먼저 생각

인덱스 뜻: PRCP(강수량), SNOW(적설량), SNWD(쌓인 눈 깊이), TMAX/TMIN/TOBS(최고/최저/관측 기온), WESF(눈의 수분량), inclement_weather(악천후 여부)

In [ ]:
import pandas as pd

# 일부러 문제들만 있는 존나 이상한 데이터를 가져옴 그래서 dirty data임
df = pd.read_csv('data/dirty_data.csv')


## Finding problematic data
A good first step is to look at some rows:

In [ ]:
# 일단 위 5행 보기. 이상한 점들:
# - station이 '?' 인 행이 있음 (관측소 이름이 없음)
# - 같은 날짜(1/1)가 3번 반복됨 (중복)
# - SNWD(적설 깊이)가 -inf (마이너스 무한대??)
# - TMAX가 5505.0, TMIN이 -40.0
# -> 이런 게 이상한 값 또는 값이 없을 때 대신 넣어둔 것인 placeholder라고 불림.
df.head()


,date,station,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
0,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
1,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
2,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
3,2018-01-02T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-8.3,-16.1,-12.2,NaN,False
4,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False


Looking at summary statistics can reveal strange or missing values:

In [ ]:
# describe로 통계 보면 이상한 값이 드러남
# SNWD: min이 -inf, mean/std가 NaN -> 무한대 값이 섞여 있어서 계산 불가
# TMAX: max 5505, 75%도 5505 -> 가짜 값이 엄청 많음 (전체의 1/4 이상)
# TMIN: min -40, 25%도 -40 -> 역시 가짜 값 많음
# (RuntimeWarning은 무한대 때문에 계산이 이상하다는 경고. 무시해도 됨)
df.describe()


/home/stefaniemolin/book_env/lib/python3.7/site-packages/numpy/lib/function_base.py:3968: RuntimeWarning: invalid value encountered in multiply
  x2 = take(ap, indices_above, axis=axis) * weights_above


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF
count,765.000000,577.000000,577.0,765.000000,765.000000,398.000000,11.000000
mean,5.360392,4.202773,NaN,2649.175294,-15.914379,8.632161,16.290909
std,10.002138,25.086077,NaN,2744.156281,24.242849,9.815054,9.489832
min,0.000000,0.000000,-inf,-11.700000,-40.000000,-16.100000,1.800000
25%,0.000000,0.000000,NaN,13.300000,-40.000000,0.150000,8.600000
50%,0.000000,0.000000,NaN,32.800000,-11.100000,8.300000,19.300000
75%,5.800000,0.000000,NaN,5505.000000,6.700000,18.300000,24.900000
max,61.700000,229.000000,inf,5505.000000,23.900000,26.100000,28.700000


The `info()` method can pinpoint missing values and wrong data types:

In [ ]:
# info로 빈칸(Non-Null Count) 확인
# 765행인데 SNOW/SNWD는 577개, TOBS 398개, WESF 11개, inclement_weather 408개만 값이 있음 -> 빈칸 많음
# date가 object(문자열) -> 날짜 타입으로 바꿔야 함
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               765 non-null    object 
 1   station            765 non-null    object 
 2   PRCP               765 non-null    float64
 3   SNOW               577 non-null    float64
 4   SNWD               577 non-null    float64
 5   TMAX               765 non-null    float64
 6   TMIN               765 non-null    float64
 7   TOBS               398 non-null    float64
 8   WESF               11 non-null     float64
 9   inclement_weather  408 non-null    object 
dtypes: float64(7), object(3)
memory usage: 59.9+ KB


We can use the `isna()`/`isnull()` method of the series to find nulls:

그 방지민 있는 이즈나랑 스펠링 같아요~

In [ ]:
# .isna() -> 각 칸이 빈칸(NaN)인가? True/False. (isnull()과 완전히 같은 거. 이름만 두 개)
# | (OR)로 연결 -> SNOW, SNWD, TOBS, WESF, inclement_weather 중 하나라도 빈칸인 행
# 조건이 길어서 두 줄로 나눠 씀 (대괄호 안이라 줄바꿈 가능)
contain_nulls = df[
    df.SNOW.isna() | df.SNWD.isna() | df.TOBS.isna()
    | df.WESF.isna() | df.inclement_weather.isna()
]
# .shape[0] -> shape는 (행, 열)이니까 [0]은 행 개수 -> 765 = 전체 행. 즉 모든 행에 빈칸이 하나 이상 있음!
contain_nulls.shape[0]


765

In [ ]:
# 빈칸 있는 행 10개 보기. 특히 WESF, inclement_weather 열에 NaN이 많음
contain_nulls.head(10)


,date,station,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
0,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
1,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
2,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
3,2018-01-02T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-8.3,-16.1,-12.2,NaN,False
4,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False
5,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False
6,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False
7,2018-01-04T00:00:00,?,20.6,229.0,inf,5505.0,-40.0,NaN,19.3,True
8,2018-01-04T00:00:00,?,20.6,229.0,inf,5505.0,-40.0,NaN,19.3,True
9,2018-01-05T00:00:00,?,0.3,NaN,NaN,5505.0,-40.0,NaN,NaN,NaN


Note that we can't check if we have `NaN` like this:

In [ ]:
# NaN을 찾으려고 == 'NaN'(문자열)으로 비교하면? -> 0개. 안 됨!
# NaN은 'NaN'이라는 글자가 아니라 특별한 '값 없음' 표시이기 때문
df[df.inclement_weather == 'NaN'].shape[0]


0

This is because it is actually `np.nan`. However, notice this also doesn't work:

In [ ]:
import numpy as np
# 그럼 진짜 NaN인 np.nan과 == 로 비교하면? -> 역시 0개!
# NaN은 자기 자신과도 같지 않다고 정의되어 있음 (np.nan == np.nan 은 False). 파이썬의 특이한 규칙
df[df.inclement_weather == np.nan].shape[0]


0

We have to use one of the methods discussed earlier for this to work:

In [ ]:
# 그래서 NaN 찾을 땐 반드시 .isna() (또는 .isnull()) 을 써야 함 -> 357개
df[df.inclement_weather.isna()].shape[0]


357

We can find `-inf`/`inf` by comparing to `-np.inf`/`np.inf`:

In [ ]:
# inf = infinity(무한대). np.inf(양의 무한대), -np.inf(음의 무한대)
# .isin([-np.inf, np.inf]) -> 값이 둘 중 하나인가? -> SNWD에 무한대가 577개
df[df.SNWD.isin([-np.inf, np.inf])].shape[0]


577

Rather than do this for each column, we can write a function that will use a [dictionary comprehension](https://www.python.org/dev/peps/pep-0274/) to check all the columns for us:

In [ ]:
# 열마다 무한대 개수를 세는 함수 만들기 (def로 함수 정의, python_101 참고)
def get_inf_count(df):
    """Find the number of inf/-inf values per column in the dataframe"""
    # {열이름: 개수 for 열 in 모든 열} -> 딕셔너리 컴프리헨션
    # df[df[col].isin([np.inf, -np.inf])].shape[0] -> 그 열이 무한대인 행 개수
    return {
        col: df[df[col].isin([np.inf, -np.inf])].shape[0] for col in df.columns
    }

# 함수 실행 -> SNWD만 577개, 나머지는 0
get_inf_count(df)


{'date': 0,
 'station': 0,
 'PRCP': 0,
 'SNOW': 0,
 'SNWD': 577,
 'TMAX': 0,
 'TMIN': 0,
 'TOBS': 0,
 'WESF': 0,
 'inclement_weather': 0}

Before we can decide how to handle the infinite values of snow depth, we should look at the summary statistics for snowfall, which forms a big part in determining the snow depth:

In [ ]:
# SNWD가 +inf인 날 vs -inf인 날, 각각 SNOW(적설량)는 어땠는지 비교
# df[df.SNWD == np.inf].SNOW.describe() -> SNWD가 +inf(양의 무한대)인 행들의 SNOW 통계
# 두 결과를 딕셔너리로 묶어 데이터프레임 만들고 .T로 뒤집어서 보기 좋게
# 결과: +inf인 날은 눈이 많이 옴(평균 101mm), -inf인 날은 눈이 0
# -> 아마 +inf는 "눈이 너무 많이 쌓여서 측정 불가", -inf는 "눈 없음"을 뜻하는 것 같다고 추측할 수 있음
pd.DataFrame({
    'np.inf Snow Depth': df[df.SNWD == np.inf].SNOW.describe(),
    '-np.inf Snow Depth': df[df.SNWD == -np.inf].SNOW.describe()
}).T


,count,mean,std,min,25%,50%,75%,max
np.inf Snow Depth,24.0,101.041667,74.498018,13.0,25.0,120.5,152.0,229.0
-np.inf Snow Depth,553.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


Let's now look into the `date` and `station` columns. We saw the `?` for station earlier, so we know that was the other unique value. However, we see that some dates are present 8 times in the data and we only have 324 days meaning we are also missing days:

In [ ]:
# 문자열 열들만 describe
# date: 765개인데 unique(서로 다른 값)가 324개 -> 날짜가 중복됨. 가장 많은 날짜(top)는 8번(freq)이나 나옴
#   1년은 365일인데 324일뿐 -> 빠진 날도 있음
# station: 2종류 (진짜 관측소 이름, 그리고 '?')
df.describe(include='object')


,date,station,inclement_weather
count,765,765,408
unique,324,2,2
top,2018-07-05T00:00:00,GHCND:USC00280907,False
freq,8,398,384


We can use the `duplicated()` method to find duplicate rows:

In [ ]:
# .duplicated() -> 각 행이 '앞에 나온 행과 완전히 똑같은가?' True/False
# df[그 마스크] -> 중복인 행만 -> 284개
df[df.duplicated()].shape[0]


284

The default for `keep` is `'first'` meaning it won't show the first row that the duplicated data was seen in; we can pass in `False` to see it though:

In [ ]:
# keep=False -> 중복된 행은 '첫 번째까지 포함해서' 전부 표시
# 기본값 keep='first'는 첫 번째는 원본으로 보고 True로 안 잡음
# 284 -> 482로 늘어남 (첫 번째 것들도 포함됐으니까)
df[df.duplicated(keep=False)].shape[0]


482

We can also specify the columns to use:

In [ ]:
# .duplicated(['date', 'station']) -> 모든 열이 아니라 date와 station 두 열만 같으면 중복으로 판단
# 결과가 284로 같음 -> 여기서는 date+station이 같으면 나머지도 다 같다는 뜻
df[df.duplicated(['date', 'station'])].shape[0]


284

Let's look at a few duplicates. Just in the few values we see here, we know that the top 4 are actually in the data 6 times because by default we aren't seeing their first occurrence:

In [ ]:
# 중복 행 5개 보기. 인덱스 1, 2 (1/1의 2, 3번째), 5, 6 (1/3의 2, 3번째)...
# 첫 번째 것(인덱스 0, 4)은 안 보임 (keep='first' 기본값)
df[df.duplicated()].head()


,date,station,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
1,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
2,2018-01-01T00:00:00,?,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
5,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False
6,2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False
8,2018-01-04T00:00:00,?,20.6,229.0,inf,5505.0,-40.0,NaN,19.3,True


## Mitigating Issues

### Handling duplicated data
Since we know we have NY weather data and noticed we only had two entries for `station`, we may decide to drop the `station` column because we are only interested in the weather data. However, when dealing with duplicate data, we need to think of the ramifications of removing it. Notice we only have data for the `WESF` column when the station is `?`:

In [ ]:
# df[df.WESF.notna()] -> WESF(눈의 수분량)에 값이 있는 행만
# .station.unique() -> 그 행들의 관측소 종류 -> '?' 만 나옴!
# 즉 WESF 데이터는 '?' 관측소에만 있음. station이 '?'라고 무작정 지우면 WESF를 전부 잃게 됨 -> 조심해야 함
df[df.WESF.notna()].station.unique()


array(['?'], dtype=object)

If we determine it won't impact our analysis, we can use `drop_duplicates()` to remove them:

In [ ]:
# 중복 제거 방법 6단계. 그냥 drop_duplicates 하면 WESF를 잃으니까 머리를 좀 씀

# 1. make the date a datetime (날짜를 날짜 타입으로)
df.date = pd.to_datetime(df.date)

# 2. save this information for later
# '?' 관측소의 WESF 값을 따로 저장해둠. '?'행만 -> 날짜 중복 제거 -> date를 인덱스로 -> WESF 열만 (시리즈)
station_qm_wesf = df[df.station == '?'].drop_duplicates('date').set_index('date').WESF

# 3. sort ? to the bottom
# station 기준 내림차순 정렬 -> 'G'로 시작하는 진짜 관측소가 위로, '?'가 아래로 (문자 순서상 G가 ?보다 뒤)
df.sort_values('station', ascending=False, inplace=True)

# 4. drop duplicates based on the date column keeping the first occurrence
# which will be the valid station if it has data
# .drop_duplicates('date') -> 날짜가 같으면 첫 번째만 남김. 위에서 정렬해놨으니 첫 번째 = 진짜 관측소
df_deduped = df.drop_duplicates('date')

# 5. remove the station column because we are done with it
# station 열 삭제 -> date를 인덱스로 -> 날짜순 정렬
df_deduped = df_deduped.drop(columns='station').set_index('date').sort_index()

# 6. take valid station's WESF and fall back on station ? if it is null
# .combine_first(다른시리즈) -> 내 값이 있으면 내 값, 내 값이 NaN이면 다른 시리즈 값으로 채움
# 진짜 관측소 WESF가 비어있으면 2단계에서 저장한 '?' 관측소 WESF로 채움 (인덱스=날짜로 짝 맞춰서)
df_deduped = df_deduped.assign(
    WESF=lambda x: x.WESF.combine_first(station_qm_wesf)
)

# 765행 -> 324행 (날짜 하나에 한 행씩), station 빼서 열 8개
df_deduped.shape


(324, 8)

Here we used the `combine_first()` method to coalesce the values to the first non-null entry; this means that if we had data from both stations, we would first take the value provided by the named station and if (and only if) that station was null would we take the value from the station named `?`. The following table contains some examples of how this would play out:

| station GHCND:USC00280907 | station ? | result of `combine_first()` |
| :---: | :---: | :---: |
| 1 | 17 | 1 |
| 1 | `NaN` | 1 |
| `NaN` | 17 | 17 |
| `NaN` | `NaN` | `NaN` |

Check out the 4th row&mdash;we have `WESF` in the correct spot thanks to the index:

In [ ]:
# 결과 확인. 1/4행의 WESF에 19.3이 들어옴 -> '?' 관측소 값이 잘 합쳐진 거임
df_deduped.head()


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,-inf,5505.0,-40.0,NaN,NaN,NaN
2018-01-02,0.0,0.0,-inf,-8.3,-16.1,-12.2,NaN,False
2018-01-03,0.0,0.0,-inf,-4.4,-13.9,-13.3,NaN,False
2018-01-04,20.6,229.0,inf,5505.0,-40.0,NaN,19.3,True
2018-01-05,14.2,127.0,inf,-4.4,-13.9,-13.9,NaN,True


### Dealing with nulls
We could drop nulls, replace them with some arbitrary value, or impute them using the surrounding data. Each of these options may have ramifications, so we must choose wisely.

We can use `dropna()` to drop rows where any column has a null value. The default options leave us hardly any data:

In [ ]:
# .dropna() -> 빈칸(NaN)이 하나라도 있는 행을 삭제
# 기본 설정으로 하면 324행 중 4행만 남음 -> 데이터가 거의 다 날아감. 함부로 쓰면 안 됨!
df_deduped.dropna().shape


(4, 8)

If we pass `how='all'`, we can choose to only drop rows where everything is null, but this removes nothing:

In [ ]:
# how='all' -> 모든 열이 전부 NaN인 행만 삭제 (기본값 how='any'는 하나라도 NaN이면 삭제)
# 전부 NaN인 행은 없어서 324행 그대로
df_deduped.dropna(how='all').shape


(324, 8)

We can use just a subset of columns to determine what to drop with the `subset` argument:

In [ ]:
# subset=[열들] -> 이 열들만 보고 판단. inclement_weather, SNOW, SNWD 가 전부 NaN인 행만 삭제
# -> 293행 남음 (31행 삭제)
df_deduped.dropna(
    how='all', subset=['inclement_weather', 'SNOW', 'SNWD']
).shape


(293, 8)

This can also be performed along columns, and we can also require a certain number of null values before we drop the data:

In [ ]:
# axis='columns' -> 행이 아니라 '열'을 삭제
# thresh=행의 수 * 0.75 -> 값이 있는 칸이 전체의 75% 이상인 열만 남겨 (thresh = threshold 기준선)
# WESF는 값이 너무 적어서 삭제됨. .columns로 남은 열 확인
df_deduped.dropna(axis='columns', thresh=df_deduped.shape[0] * .75).columns


Index(['PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN', 'TOBS', 'inclement_weather'], dtype='object')

We can choose to fill in the null values instead with `fillna()`:

In [ ]:
# .fillna(0) -> 빈칸을 0으로 채움 (fill = 채우다, na = 빈값)
# WESF(눈의 수분량)가 비어있으면 눈이 안 온 거라고 보고 0으로
# inplace=True -> 원본 직접 수정
df_deduped.loc[:,'WESF'].fillna(0, inplace=True)
df_deduped.head()


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,-inf,5505.0,-40.0,NaN,0.0,NaN
2018-01-02,0.0,0.0,-inf,-8.3,-16.1,-12.2,0.0,False
2018-01-03,0.0,0.0,-inf,-4.4,-13.9,-13.3,0.0,False
2018-01-04,20.6,229.0,inf,5505.0,-40.0,NaN,19.3,True
2018-01-05,14.2,127.0,inf,-4.4,-13.9,-13.9,0.0,True


At this point we have done everything we can without distorting the data. We know that we are missing dates, but if we reindex, we don't know how to fill in the `NaN` data. With the weather data, we can't assume because it snowed one day that it will snow the next or that the temperature will be the same. For this reason, note that the next few examples are just for illustrative purposes only—just because we can do something doesn't mean we should.

That being said, let's try to address some of remaining issues with the temperature data. We know that when `TMAX` is the temperature of the Sun, it must be because there was no measured value, so let's replace it with `NaN`. We will also do so for `TMIN` which currently uses -40°C for its placeholder when we know that the coldest temperature ever recorded in NYC was -15°F (-26.1°C) on February 9, 1934:

In [ ]:
# .replace(찾을값, 바꿀값) -> TMAX의 5505(태양 온도, 가짜 값)를 NaN으로, TMIN의 -40(가짜 값)을 NaN으로
# 말도 안 되는 값은 차라리 "값 없음"으로 정직하게 표시하는 게 낫다고 하여 이렇게 표시함
df_deduped = df_deduped.assign(
    TMAX=lambda x: x.TMAX.replace(5505, np.nan),
    TMIN=lambda x: x.TMIN.replace(-40, np.nan),
)


We will also make an assumption that the temperature won't change drastically day-to-day. Note that this is actually a big assumption, but it will allow us to understand how `fillna()` works when we provide a strategy through the `method` parameter. The `fillna()` method gives us 2 options for the `method` parameter:
- `'ffill'` to forward-fill
- `'bfill'` to back-fill

*Note that `'nearest'` is missing because we are not reindexing.*

Here, we will use `'ffill'` to show how this works:

In [ ]:
# .fillna(method='ffill') -> 빈칸을 바로 앞(위) 행 값으로 채움 (forward fill = 앞으로 채우기)
# 1/4의 TMAX가 NaN이었는데 1/3 값(-4.4)으로 채워짐
# 1/1은 앞에 아무것도 없어서 그대로 NaN
# (method='bfill'은 반대로 뒤(아래) 값으로 채움 = back fill)
df_deduped.assign(
    TMAX=lambda x: x.TMAX.fillna(method='ffill'),
    TMIN=lambda x: x.TMIN.fillna(method='ffill')
).head()


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,-inf,NaN,NaN,NaN,0.0,NaN
2018-01-02,0.0,0.0,-inf,-8.3,-16.1,-12.2,0.0,False
2018-01-03,0.0,0.0,-inf,-4.4,-13.9,-13.3,0.0,False
2018-01-04,20.6,229.0,inf,-4.4,-13.9,NaN,19.3,True
2018-01-05,14.2,127.0,inf,-4.4,-13.9,-13.9,0.0,True


We can use `np.nan_to_num()` to turn `np.nan` into 0 and `-np.inf`/`np.inf` into large negative or positive finite numbers:

In [ ]:
# np.nan_to_num() -> NaN은 0으로, -inf/inf는 컴퓨터가 표현할 수 있는 가장 작은/큰 숫자로 바꿈
# SNWD의 -inf가 -1.797693e+308 (= -1.79 × 10의 308제곱) 같은 엄청 큰 숫자로 바뀜
# 무한대는 없어졌지만 값이 말이 안 되는 건 그대로라서 여기선 좋은 방법은 아님
df_deduped.assign(
    SNWD=lambda x: np.nan_to_num(x.SNWD)
).head()


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,-1.797693e+308,NaN,NaN,NaN,0.0,NaN
2018-01-02,0.0,0.0,-1.797693e+308,-8.3,-16.1,-12.2,0.0,False
2018-01-03,0.0,0.0,-1.797693e+308,-4.4,-13.9,-13.3,0.0,False
2018-01-04,20.6,229.0,1.797693e+308,NaN,NaN,NaN,19.3,True
2018-01-05,14.2,127.0,1.797693e+308,-4.4,-13.9,-13.9,0.0,True


Depending on the data we are working with, we can use the `clip()` method as an alternative to `np.nan_to_num()`. The `clip()` method makes it possible to cap values at a specific minimum and/or maximum threshold. Since `SNWD` can't be negative, let's use `clip()` to enforce a lower bound of zero. To show how the upper bound works, let's use the value of `SNOW`:

In [ ]:
# .clip(최소, 최대) -> 범위를 벗어난 값을 잘라냄. 최소보다 작으면 최소로, 최대보다 크면 최대로
# SNWD.clip(0, x.SNOW) -> 적설 깊이는 0 아래로 못 가고, 그날 적설량(SNOW)보다 클 수 없게
# -inf -> 0으로, inf -> 그날 SNOW 값으로. 훨씬 말이 되는 값!
df_deduped.assign(
    SNWD=lambda x: x.SNWD.clip(0, x.SNOW)
).head()


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN
2018-01-02,0.0,0.0,0.0,-8.3,-16.1,-12.2,0.0,False
2018-01-03,0.0,0.0,0.0,-4.4,-13.9,-13.3,0.0,False
2018-01-04,20.6,229.0,229.0,NaN,NaN,NaN,19.3,True
2018-01-05,14.2,127.0,127.0,-4.4,-13.9,-13.9,0.0,True


We can couple `fillna()` with other types of calculations. Here we replace missing values of `TMAX` with the median of all `TMAX` values, `TMIN` with the median of all `TMIN` values, and `TOBS` to the average of the `TMAX` and `TMIN` values. Since we place `TOBS` last, we have access to the imputed values for `TMIN` and `TMAX` in the calculation:

In [ ]:
# 빈칸을 계산한 값으로 채우기
# TMAX 빈칸 -> TMAX 전체의 중앙값(median)으로 / TMIN 빈칸 -> TMIN 중앙값으로
# TOBS 빈칸 -> (TMAX + TMIN) / 2 = 그날 최고/최저 평균으로
# TOBS를 마지막에 썼기 때문에 방금 채운 TMAX, TMIN 값을 쓸 수 있음 (lambda x 덕분. 순서 중요!)
df_deduped.assign(
    TMAX=lambda x: x.TMAX.fillna(x.TMAX.median()),
    TMIN=lambda x: x.TMIN.fillna(x.TMIN.median()),
    # average of TMAX and TMIN
    TOBS=lambda x: x.TOBS.fillna((x.TMAX + x.TMIN) / 2)
).head()


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,-inf,14.4,5.6,10.0,0.0,NaN
2018-01-02,0.0,0.0,-inf,-8.3,-16.1,-12.2,0.0,False
2018-01-03,0.0,0.0,-inf,-4.4,-13.9,-13.3,0.0,False
2018-01-04,20.6,229.0,inf,14.4,5.6,10.0,19.3,True
2018-01-05,14.2,127.0,inf,-4.4,-13.9,-13.9,0.0,True


We can also use `apply()` for running the same calculation across columns. For example, let's fill all missing values with their rolling 7-day median of their values, setting the number of periods required for the calculation to 0 to ensure we don't introduce more extra `NaN` values. Rolling calculations will be covered in chapter 4, so this is a preview:

In [ ]:
# .apply(함수) -> 모든 열에 같은 함수를 적용. x에 열이 하나씩 들어옴
# x.rolling(7, min_periods=0).median() -> '최근 7일'씩 묶어서(rolling = 굴러가는 창) 중앙값 계산
#   min_periods=0 -> 7일이 안 채워져도(처음 며칠) 계산해줘
# x.fillna(...) -> 빈칸을 그 7일 중앙값으로 채움
# 1/4 TMAX가 -6.35로 채워짐 = 그 전 값들(-8.3, -4.4)의 중앙값
df_deduped.apply(
    # rolling calculations will be covered in chapter 4, this is a rolling 7-day median
    # we set min_periods (# of periods required for calculation) to 0 so we always get a result
    lambda x: x.fillna(x.rolling(7, min_periods=0).median())
).head(10)


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01,0.0,0.0,-inf,NaN,NaN,NaN,0.0,NaN
2018-01-02,0.0,0.0,-inf,-8.30,-16.1,-12.20,0.0,False
2018-01-03,0.0,0.0,-inf,-4.40,-13.9,-13.30,0.0,False
2018-01-04,20.6,229.0,inf,-6.35,-15.0,-12.75,19.3,True
2018-01-05,14.2,127.0,inf,-4.40,-13.9,-13.90,0.0,True
2018-01-06,0.0,0.0,-inf,-10.00,-15.6,-15.00,0.0,False
2018-01-07,0.0,0.0,-inf,-11.70,-17.2,-16.10,0.0,False
2018-01-08,0.0,0.0,-inf,-7.80,-16.7,-8.30,0.0,False
2018-01-10,0.0,0.0,-inf,5.00,-7.8,-7.80,0.0,False


The last strategy we could try is interpolation with the `interpolate()` method. We specify the `method` parameter with the interpolation strategy to use. There are many options, but we will stick with the default of `'linear'`, which will treat values as evenly spaced and place missing values in the middle of existing ones. We have some missing data, so we will reindex first. Look at January 9th, which we didn't have before—the values for `TMAX`, `TMIN`, and `TOBS` are the average of values the day prior (January 8th) and the day after (January 10th):

In [ ]:
# .reindex(pd.date_range('2018-01-01', '2018-12-31', freq='D')) -> 1/1~12/31 매일로 인덱스를 맞춤. 빠진 날짜는 NaN 행으로 생김
# .apply(lambda x: x.interpolate()) -> 모든 열에 interpolate(보간) 적용
#   interpolate = 앞뒤 값 사이를 직선으로 이어서 중간값을 추정. 앞이 10, 뒤가 20이면 가운데는 15
# \ 는 줄 이어쓰기 표시
# 1/4 TOBS가 -13.6 = (1/3의 -13.3 + 1/5의 -13.9) / 2 -> 앞뒤 평균으로 채워진 거임
# 원래 없던 1/9 행도 생기고 1/8, 1/10의 평균으로 채워짐
df_deduped\
    .reindex(pd.date_range('2018-01-01', '2018-12-31', freq='D'))\
    .apply(lambda x: x.interpolate())\
    .head(10)


,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WESF,inclement_weather
2018-01-01,0.0,0.0,-inf,NaN,NaN,NaN,0.0,NaN
2018-01-02,0.0,0.0,-inf,-8.3,-16.10,-12.20,0.0,False
2018-01-03,0.0,0.0,-inf,-4.4,-13.90,-13.30,0.0,False
2018-01-04,20.6,229.0,inf,-4.4,-13.90,-13.60,19.3,True
2018-01-05,14.2,127.0,inf,-4.4,-13.90,-13.90,0.0,True
2018-01-06,0.0,0.0,-inf,-10.0,-15.60,-15.00,0.0,False
2018-01-07,0.0,0.0,-inf,-11.7,-17.20,-16.10,0.0,False
2018-01-08,0.0,0.0,-inf,-7.8,-16.70,-8.30,0.0,False
2018-01-09,0.0,0.0,-inf,-1.4,-12.25,-8.05,0.0,NaN
2018-01-10,0.0,0.0,-inf,5.0,-7.80,-7.80,0.0,False


<hr>

<div style="overflow: hidden; margin-bottom: 10px;">
    <div style="float: left;">
         <a href="./4-reshaping_data.ipynb">
            <button>&#8592; Previous Notebook</button>
        </a>
    </div>
    <div style="float: right;">
        <a href="../../solutions/ch_03/solutions.ipynb">
            <button>Solutions</button>
        </a>
        <a href="../ch_04/1-querying_and_merging.ipynb">
            <button>Chapter 4 &#8594;</button>
        </a>
    </div>
</div>
<hr>